# NASDAQ-100 Microstructure: Holdout Backtest

**Chapter 20 - Out-of-sample evaluation**

[`18_holdout_predictions`](18_holdout_predictions.ipynb) refitted the selected
configuration on the history before the holdout window and wrote its predictions over it.
This notebook trades them, with the concentration, the allocator, the risk overlay and the
cost assumption the rest of the case study used, and registers the result.

Nothing is chosen here. The predictions, the allocator, the concentration, the rebalance
cadence, the overlay and the charge all arrive fixed from earlier notebooks, and the only
thing this notebook decides is that they are applied unchanged. That is the whole design: a
holdout result is worth something exactly to the extent that no decision was made after
seeing it, and every knob left open here would be a decision.

The comparison to validation is printed but not interpreted. The validation figure is the
maximum of a ranking over more than a thousand backtests and the holdout figure is one
measurement over a much shorter window; what can be said about the gap between them is
[`20_strategy_analysis`](20_strategy_analysis.ipynb)'s subject, with the intervals to say
it.

**Prerequisites:** [`18_holdout_predictions`](18_holdout_predictions.ipynb).

**Scope:** one backtest. No selection, no comparison beyond a printed pair.

<!-- ML4T_ZH_BEGIN c0000-p000 -->
# NASDAQ-100 微觀結構：保留測試期回測

**第 20 章 — 樣本外評估**

[`18_holdout_predictions`](18_holdout_predictions.ipynb) 已使用保留測試期之前的歷史資料重新訓練選定設定，並產生該期間的預測。本 Notebook 依照案例研究先前使用的集中程度、配置方法、風險覆蓋規則與成本假設交易這些預測，並登錄結果。

此處不作任何選擇。預測、配置方法、集中程度、再平衡頻率、覆蓋規則及成本都已由先前 Notebook 固定；本 Notebook 唯一決定的是原樣套用它們。這就是整體設計：保留測試期結果的價值，取決於看到結果後沒有再作決策；此處任何未固定的調整參數都會構成一項新決策。

與驗證期的比較會列印出來，但不在此解讀。驗證期數字是超過一千次回測的排名最大值，而保留測試期數字來自短得多的期間中的單次量測。兩者差距能說明什麼，須由 [`20_strategy_analysis`](20_strategy_analysis.ipynb) 結合區間估計討論。

**先備條件：**[`18_holdout_predictions`](18_holdout_predictions.ipynb)。

**範圍：**一次回測。除列印兩個數字外，不作選擇或比較。
<!-- ML4T_ZH_END c0000-p000 -->


In [1]:
"""NASDAQ-100 Microstructure: Holdout Backtest."""

import json
import sqlite3

import polars as pl

from case_studies.research import open_study
from case_studies.research.holdout import build_holdout_training_spec
from case_studies.utils.backtest_loaders import get_backtest_config, load_backtest_prices_for
from case_studies.utils.backtest_presets import ensure_backtest_spec, strategy_view
from case_studies.utils.backtest_runner import resolved_allow_short_selling, run_backtest
from case_studies.utils.conformal import (
    compute_holdout_conformal_widths,
    ensure_conformal_calibration_identity,
    holdout_conformal_embargo_steps,
)
from case_studies.utils.registry import (
    backtest_run_status,
    read_predictions,
    training_hash_from_spec,
)
from case_studies.utils.strategy_analysis import resolve_solvent_carrier
from utils.paths import get_case_study_dir

In [2]:
CASE_STUDY_ID = "nasdaq100_microstructure"
EXECUTION_TIER = "canonical"
WORKSPACE: str = ""
MAX_SYMBOLS = 0

In [3]:
study = open_study(CASE_STUDY_ID, execution_tier=EXECUTION_TIER, workspace=WORKSPACE or None)
CASE_DIR = get_case_study_dir(CASE_STUDY_ID)
bt_config = get_backtest_config(CASE_STUDY_ID)

# `MAX_SYMBOLS` reduces the price panel this run trades and reaches `backtest_hash` through
# nothing, so a reduced run and a full run over the same holdout predictions hash alike and
# the second is served the first's result (ml4t/agent-workspace#911). `14_backtest` and
# `15_portfolio_management` give a reduced run an identity of its own by declaring the traded
# universe into the spec they BUILD; this notebook carries the selected configuration's spec
# forward through `ensure_backtest_spec`, which has no such parameter, so there is no identity
# to give one here. That leaves refusal as the only correct answer on the canonical tier, and
# it matters more here than anywhere else in the case study: this is the one window the study
# reports as unseen, it carries one backtest, and a narrowed result registered against it
# could not be distinguished afterwards from the declared portfolio's.
if EXECUTION_TIER == "canonical" and MAX_SYMBOLS:
    raise ValueError(
        "MAX_SYMBOLS narrows the universe this run trades, which makes it a different "
        "portfolio from the declared one and gives it its own backtest identity "
        "(ml4t/agent-workspace#911). A canonical run trades the declared universe: set "
        "MAX_SYMBOLS=0, or run under EXECUTION_TIER='preview' with a WORKSPACE."
    )


def _registered_holdout_backtests(case_dir, prediction_hash):
    """The backtest hashes already registered against one holdout prediction set."""
    with sqlite3.connect(str(case_dir / "run_log" / "registry.db")) as conn:
        rows = conn.execute(
            "SELECT backtest_hash FROM backtest_runs WHERE prediction_hash = ? "
            "ORDER BY backtest_hash",
            (prediction_hash,),
        ).fetchall()
    return [backtest_hash for (backtest_hash,) in rows]

## 1. The configuration, and the predictions it produced on the holdout

The selected configuration is resolved the same way [`17_costs`](17_costs.ipynb) and
[`18_holdout_predictions`](18_holdout_predictions.ipynb) resolve it, so all three run the
same configuration by construction rather than by a hash copied between them.

Which holdout prediction set belongs to it is derived rather than searched for. Re-deriving the
holdout training specification reproduces the training identity 18 registered - the derivation is
deterministic and the identity covers it - so the prediction set is looked up by that identity
and the selected configuration's checkpoint. A search over holdout prediction sets would have to
guess which one belonged to this configuration.

<!-- ML4T_ZH_BEGIN c0004-p000 -->
## 1. 選定設定及其在保留測試期產生的預測

選定設定的解析方法與 [`17_costs`](17_costs.ipynb) 和 [`18_holdout_predictions`](18_holdout_predictions.ipynb) 相同，因此三者因設計而使用相同設定，而非靠彼此複製雜湊值。

屬於此設定的保留測試期預測集是推導出來的，不是搜尋出來的。重新推導保留測試期訓練規格，會重現第 18 份 Notebook 登錄的訓練身分：推導過程是確定性的，身分也涵蓋該過程；因此可用此身分及選定設定的檢查點查找預測集。若改為搜尋保留測試期預測集，就必須猜測哪一組屬於此設定。
<!-- ML4T_ZH_END c0004-p000 -->


In [4]:
carrier = resolve_solvent_carrier(CASE_STUDY_ID)
LABEL = carrier["label"]
validation_prediction_record = study.results.open(carrier["val_prediction_hash"]).registry_record()

holdout_spec = build_holdout_training_spec(
    study,
    study.results.open(carrier["training_hash"]).spec(),
    timeline=(
        pl.read_parquet(study.root / "labels" / f"{LABEL}.parquet")
        .get_column("timestamp")
        .unique()
        .sort()
        .to_list()
    ),
    case_study=CASE_STUDY_ID,
)
holdout_training_hash = training_hash_from_spec(holdout_spec)

with sqlite3.connect(str(CASE_DIR / "run_log" / "registry.db")) as conn:
    match = conn.execute(
        """
        SELECT prediction_hash FROM prediction_sets
        WHERE split = 'holdout' AND training_hash = ?
          AND checkpoint_kind IS ? AND checkpoint_value IS ?
        """,
        (
            holdout_training_hash,
            validation_prediction_record["checkpoint_kind"],
            validation_prediction_record["checkpoint_value"],
        ),
    ).fetchone()
if match is None:
    # Two causes, and they call for opposite actions, so the message has to tell them apart.
    # 18 not having run is the obvious one. The other is that the carrier MOVED between 18 and
    # this notebook: both resolve independently and by design, and this case study declares no
    # backtest population, so `published_members_at(member_kind="backtest")` is None and no
    # backtest row is ever retired from the pool. A sweep that registers a higher-Sharpe cell
    # between the two runs therefore changes what `resolve_solvent_carrier` returns, this
    # notebook re-derives a different training identity, and the lookup misses.
    #
    # Sending that case to "run 18 first" points at the wrong thing. 18 would refuse it - its
    # `holdout_generations_to_retire` check exists for exactly this - but only after the reader
    # has spent a cycle being told to do the thing that cannot work.
    with sqlite3.connect(str(CASE_DIR / "run_log" / "registry.db")) as conn:
        registered = conn.execute(
            """
            SELECT t.config_name, t.label, p.training_hash,
                   p.checkpoint_kind, p.checkpoint_value
            FROM prediction_sets p JOIN training_runs t ON t.training_hash = p.training_hash
            WHERE p.split = 'holdout'
            ORDER BY p.created_at
            """
        ).fetchall()
    # Checkpoint is part of the generation, not a detail below it: 18 keys its retirement
    # check on `(training_hash, (checkpoint_kind, checkpoint_value))`, and the lookup above
    # matches on all three. So a set for this configuration at a DIFFERENT checkpoint is a
    # different configuration and belongs in the moved-selection branch, not the replay one.
    _ckpt = (
        validation_prediction_record["checkpoint_kind"],
        validation_prediction_record["checkpoint_value"],
    )
    same_config = [
        (cfg, lab, th)
        for cfg, lab, th, ck_kind, ck_value in registered
        if cfg == carrier["config_name"] and lab == LABEL and (ck_kind, ck_value) == _ckpt
    ]
    named = ", ".join(
        f"{cfg} on {lab} at {ck_kind}={ck_value} ({th})"
        for cfg, lab, th, ck_kind, ck_value in registered[:4]
    )
    if not registered:
        msg = (
            f"No holdout prediction set for training {holdout_training_hash}, and this registry "
            "holds none at all. Run 18_holdout_predictions first; this notebook does not fit."
        )
    elif same_config:
        # The configuration is the one that was fitted, under a training identity that is not
        # the one this notebook just derived. Selection did not move; the DERIVATION did - the
        # holdout spec, the label timeline it is built from, or something the identity hashes.
        # That diagnosis differs from the branch below, which is why they are separated, but
        # the action does not: 18 refuses in both cases. `holdout_generations_to_retire`
        # skips only a row whose (training_hash, checkpoint) equals the generation about to
        # be registered, so a row under the earlier identity is not skipped. It lands in one
        # of the three retirement buckets by its training spec - `superseded` if it was a
        # genuine refit, otherwise `not_out_of_sample` or `unattributable` - and each of the
        # three raises in 18 before anything is fitted. The `superseded` message reads "a
        # refit of a different configuration", which is 18's identity for a generation and
        # not the config name.
        msg = (
            f"No holdout prediction set for training {holdout_training_hash}, but "
            f"{carrier['family']}/{carrier['config_name']} on {LABEL} at "
            f"{_ckpt[0]}={_ckpt[1]} - the configuration that resolves now - already has one under "
            f"{', '.join(th for _, _, th in same_config)}. So the selection did not move and "
            "its derived training identity did: the holdout spec, the label timeline it reads, "
            "or an input the identity covers has changed since 18 ran. Re-running 18 on its "
            "own will NOT recover this: it matches a registered generation on the exact "
            "training hash and checkpoint, so the row under the earlier identity reads as a "
            "superseded refit and 18 refuses before fitting. Either restore the derivation "
            "the registered row was fitted under, or retire that generation through the "
            "registry's own lifecycle - which records that the window was looked at twice - "
            "and then re-run 18."
        )
    else:
        msg = (
            f"No holdout prediction set for training {holdout_training_hash}, but this registry "
            f"holds {len(registered)} for other configurations: {named}. So "
            "18_holdout_predictions has run and the selected configuration has MOVED since - it "
            f"now resolves to {carrier['family']}/{carrier['config_name']} on {LABEL}. A "
            "backtest row is never retired from the carrier pool, so a sweep registering a "
            "higher-Sharpe cell between the two notebooks is enough to do it. Do NOT re-run 18 "
            "to fit the new one: that spends a second holdout observation on a second "
            "configuration, which is what its retirement check refuses. Decide which "
            "configuration this case study carries."
        )
    raise RuntimeError(msg)
HOLDOUT_PREDICTION_HASH = match[0]

print(f"Selected configuration: {carrier['val_backtest_hash']}  {carrier['config_name']} ({LABEL})")
print(f"Holdout training:   {holdout_training_hash}")
print(f"Holdout prediction: {HOLDOUT_PREDICTION_HASH}")

Selected configuration: a2d22880e93f  default_multiclass (fwd_dir_15m)
Holdout training:   f434e268a3e5
Holdout prediction: 345655bde69d


## 2. Calibration, where the allocator needs it

An allocator that sizes by a conformal width is calibrated from errors the model has already
made, and on the holdout there are none to use: an error is usable only once the return it
measures has been realised, and every holdout return realises inside the window being evaluated.
Such a selected configuration takes its widths from the validation residuals of the validation
prediction set, which is what the allocator would have had standing at the start of the window.

The embargo matters for this case study's label and would not for every one. A residual
observed at `t` is not resolved until `t + h + 1min`, one bar PAST the label's horizon: the
entry leg is the VWAP of the bar after the decision, so a label at `t` consumes a quote at
`t + h + 1`. So the last residuals of the validation span reach into the holdout window, and
calibrating on them would size holdout positions with holdout price information.

The value comes from the reviewed table in `conformal.py`, which records the label BUFFER rather
than the horizon for exactly that reason - 6, 16 and 61 for `fwd_ret_5m`,
`fwd_ret_15m`/`fwd_dir_15m` and `fwd_ret_60m`. It is read from the selected configuration's label
rather than assumed from the primary, because the selected configuration is resolved across every
declared label. The unit is prediction-data steps, and on this panel a step is one MINUTE, not
one decision slot: the modal gap between adjacent prediction timestamps is 0:01:00 for all four
labels. Counting in decision slots is what an earlier version of that table did, and it embargoed
one minute where the label reaches sixteen.


<!-- ML4T_ZH_BEGIN c0006-p000 -->
## 2. 配置方法需要時使用的校準

若配置方法依共形區間寬度決定部位規模，它須用模型已犯過的錯誤進行校準；但保留測試期沒有可用的這類錯誤：只有被衡量的報酬已實現後，對應誤差才能使用，而保留測試期報酬都在評估期間內才實現。因此，這類選定設定應採用驗證期預測集的驗證殘差所產生的區間寬度；這也是在保留測試期開始時實際可得的資訊。

禁用期對本案例的標籤很重要，但不見得對所有案例都重要。於 `t` 觀察到的殘差，要到 `t + h + 1min` 才會確定，比標籤前瞻期間還晚一根時間棒：進場端使用決策後一根時間棒的 VWAP，因此 `t` 的標籤會用到 `t + h + 1` 的報價。驗證期尾端的殘差因而延伸至保留測試期間；若用它們校準，就會用保留測試期價格資訊決定該期間的部位規模。

這個數值來自 `conformal.py` 中經審核的表格；正因上述原因，表格記錄的是標籤**緩衝區**，而非前瞻期間：對 `fwd_ret_5m`、`fwd_ret_15m`／`fwd_dir_15m` 以及 `fwd_ret_60m`，數值分別為 6、16、61。該值依選定設定的標籤讀取，而非假設為主要標籤，因為選定設定會跨所有已宣告標籤解析。單位是預測資料的步數，而在此資料面板上，一步是**一分鐘**，不是一個決策時點：四個標籤各自相鄰預測時間戳記的最常見間隔都是 0:01:00。該表先前版本以決策時點計數，導致標籤影響延伸至十六分鐘時，卻只設定一分鐘禁用期。
<!-- ML4T_ZH_END c0006-p000 -->

The reach is minutes rather than the ETF study's three weeks, and it is the same defect at any
width: the residuals that cross the boundary are the ones nearest it, which are the ones a
calibration weights most.

The branch is here rather than assumed away because the selected configuration can change. It
does not fire for the one this case study currently reports, which sizes by inverse volatility
over a declared window and needs no calibration - so the line below prints that rather than
staying silent, which is what tells a reader the branch was evaluated.

The widths themselves are NOT written here. Writing them replaces the artifact an already
registered run was sized by, and the replacement guard in section 3 can still refuse this run
afterwards - which would leave the registered holdout pointing at a calibration that no
longer existed. The write is below the guard.

<!-- ML4T_ZH_BEGIN c0006-p001 -->
這裡的延伸以分鐘計，而非 ETF 研究中的三週；但不論寬度為何，缺陷相同：跨越邊界的殘差最接近邊界，也最受校準過程重視。

此分支仍保留，而非假設永遠不會用到，因為選定設定可能改變。本案例目前呈報的設定使用指定期間的反波動率來決定部位規模，不需校準，因此不會進入該分支；下方會列印這項事實，而不是保持沉默，讓讀者知道分支確實經過評估。

區間寬度本身**不在此處寫入**。寫入會取代已登錄執行結果當初據以決定部位規模的構件，而第 3 節的替換防護之後仍可能拒絕這次執行，讓已登錄的保留測試期結果指向不再存在的校準資料。因此，寫入動作安排在防護之後。
<!-- ML4T_ZH_END c0006-p001 -->


In [5]:
allocation = strategy_view(json.loads(carrier["spec_json"])).get("allocation") or {}
NEEDS_CALIBRATION = allocation.get("method") == "conformal_weighted"
embargo_steps = holdout_conformal_embargo_steps(CASE_STUDY_ID, LABEL) if NEEDS_CALIBRATION else 0
if NEEDS_CALIBRATION:
    print(f"Conformal configuration: embargo {embargo_steps} observation(s), widths written below.")
else:
    print(f"Allocator {allocation.get('method', 'equal_weight')!r} needs no calibration.")

Allocator 'equal_weight' needs no calibration.


## 3. The backtest

The strategy specification is the selected configuration's own, re-pointed at the holdout
prediction set and the holdout price window. Nothing else about it changes - the concentration,
the allocator, the risk overlay and the commission and slippage levels are the ones `setup.yaml`
declares and every validation number in this case study was net of.

The run registers under `stage='holdout'`, which the registry derives from the prediction set's
split rather than from anything asserted here - and that derivation takes precedence over the
risk block the selected configuration carries, which would otherwise file this as another risk
overlay.

**The window carries one backtest**, for the same reason 18 lets it carry one prediction
generation. 18's guard is on the model - the training identity and the checkpoint - and it
cannot see this one: a changed allocator, overlay, cost level or calibration produces the
same holdout predictions and a different result from them. Like 18's, this guard refuses
rather than offering a replacement, because deleting a result that has been seen does not
unsee it.

The test is the backtest hash, not a field-by-field comparison. Every input that changes the
result is in that hash by construction, and a guard naming fields instead has to be right
about all of them. The hash is resolved before anything runs, so nothing is evaluated on the
holdout before the question is answered, and it comes from `backtest_run_status` - the call
the runner itself makes - because asking it is the only way to be sure the guard and the
runner agree about identity. The run asserts they still agree afterwards, because a guard
that had quietly stopped predicting the hash would let everything through while looking
correct.

<!-- ML4T_ZH_BEGIN c0008-p000 -->
## 3. 回測

策略規格沿用選定設定自身的規格，只將其指向保留測試期預測集及該期間的價格資料。其餘部分完全不變：集中程度、配置方法、風險覆蓋規則、手續費與滑價水準，都與 `setup.yaml` 宣告的相同；本案例所有驗證期數字也都已扣除這些成本。

執行結果登錄於 `stage='holdout'`；登錄表會根據預測集的資料切分推導這個階段，而非依此處的宣稱。此推導優先於選定設定所帶的風險區塊，否則結果會被歸檔成另一項風險覆蓋結果。

**此期間只容許一次回測**，理由與第 18 份 Notebook 只容許一代預測相同。第 18 份的防護針對模型，即訓練身分與檢查點，無法察覺此處的改變：不同配置方法、覆蓋規則、成本水準或校準方式，都可能使用相同保留測試期預測，卻產生不同結果。與第 18 份相同，此防護會拒絕替換，而非提供替換選項，因為刪除已看過的結果，無法抹去已看過它的事實。

檢查的是回測雜湊值，而非逐欄比較。設計上，所有會改變結果的輸入都包含在雜湊值中；若防護只列出特定欄位，就必須確保清單無一遺漏。雜湊值在任何執行前解析，因此回答是否允許執行之前，不會先在保留測試期評估任何內容；它由 `backtest_run_status` 提供，也就是執行器自身會呼叫的函式，才能確保防護與執行器對身分的判定一致。執行後還會斷言兩者仍一致，因為若防護悄悄不再正確預測雜湊值，就可能看似正確，實際卻放行所有請求。
<!-- ML4T_ZH_END c0008-p000 -->


In [6]:
prices = load_backtest_prices_for(CASE_STUDY_ID, LABEL, split="holdout", max_symbols=MAX_SYMBOLS)
predictions = read_predictions(CASE_STUDY_ID, HOLDOUT_PREDICTION_HASH)
print(f"Prices: {len(prices):,} rows, {prices['symbol'].n_unique():,} symbols")
print(
    f"Predictions: {predictions.height:,} rows, "
    f"{predictions['timestamp'].n_unique():,} decision timestamps, "
    f"{predictions['timestamp'].dt.date().n_unique():,} sessions"
)

spec = ensure_backtest_spec(
    CASE_STUDY_ID,
    bt_config,
    json.loads(carrier["spec_json"]),
    prices=prices,
    prediction_hash=HOLDOUT_PREDICTION_HASH,
    initial_cash=bt_config.initial_cash,
)
spec["chapter"] = "ch20"
# The embargo goes into the specification before anything hashes it. The widths are an input to
# this backtest and the embargo decides them, so two embargoes are two results and must not
# share an identity.
if NEEDS_CALIBRATION:
    spec = ensure_conformal_calibration_identity(spec, holdout_embargo_steps=embargo_steps)

spec["backtest_config"]["account"]["allow_short_selling"] = resolved_allow_short_selling(spec, None)
prospective_hash = backtest_run_status(CASE_STUDY_ID, HOLDOUT_PREDICTION_HASH, spec).backtest_hash
superseded_backtests = sorted(
    set(_registered_holdout_backtests(CASE_DIR, HOLDOUT_PREDICTION_HASH)) - {prospective_hash}
)
if superseded_backtests:
    raise RuntimeError(
        "the holdout window already carries a backtest of a different configuration: "
        + ", ".join(superseded_backtests)
        + f". This run would register {prospective_hash} and has not run. Same rule as "
        "18_holdout_predictions: discarding the earlier result would not undo having observed "
        "it, so there is no switch here. Leave the selection where it was, or retire the "
        "earlier evaluation through the registry's lifecycle."
    )

# The guard has passed, so this run will register and the widths it is sized by are the ones
# that belong beside this prediction set.
if NEEDS_CALIBRATION:
    widths = compute_holdout_conformal_widths(
        CASE_STUDY_ID,
        carrier["val_prediction_hash"],
        HOLDOUT_PREDICTION_HASH,
        alpha=float(allocation.get("alpha", 0.2)),
        min_calibration_n=int(allocation["min_calibration_n"]),
        embargo_steps=embargo_steps,
        write=True,
    )
    print(
        f"Conformal widths: {widths.height:,} rows over "
        f"{widths['symbol'].n_unique():,} names, embargo {embargo_steps} observation(s)"
    )

result = run_backtest(
    CASE_STUDY_ID,
    HOLDOUT_PREDICTION_HASH,
    spec,
    prices=prices,
    predictions=predictions,
    label=LABEL,
    register=True,
    initial_cash=bt_config.initial_cash,
    calendar=bt_config.calendar,
)
if result.backtest_hash != prospective_hash:
    raise RuntimeError(
        f"the guard predicted {prospective_hash} and the runner registered "
        f"{result.backtest_hash}. The guard decides what may run on the holdout, so a guard "
        "that no longer reproduces the runner's identity is not a smaller problem than the one "
        "it was written for."
    )
print(f"Holdout backtest: {result.backtest_hash}")

# The stage is checked rather than trusted. This configuration comes from the risk stage and its
# spec carries a risk block, and stage inference reads the prediction's split before that block - so
# a holdout run files as `holdout`. If that order ever changes, the whole out-of-sample result lands
# in `risk_overlay` and `20_strategy_analysis` finds no holdout at all, which is a failure four
# notebooks away from its cause.
with sqlite3.connect(str(CASE_DIR / "run_log" / "registry.db")) as conn:
    registered_stage = conn.execute(
        "SELECT stage FROM backtest_runs WHERE backtest_hash = ?", (result.backtest_hash,)
    ).fetchone()[0]
if registered_stage != "holdout":
    raise RuntimeError(
        f"the holdout backtest registered under stage={registered_stage!r} rather than "
        "'holdout'; the split-based inference in registry.store._infer_stage did not take "
        "precedence over this configuration's risk block"
    )

Prices: 5,068,397 rows, 110 symbols


Predictions: 3,949,522 rows, 39,884 decision timestamps, 128 sessions


case_studies/utils/slot_strategy.py:198: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  aligned = bars.join_asof(


Holdout backtest: 7065bf7708c9


## 4. What it came out at

The two numbers below are one strategy measured on two disjoint periods, and the gap between
them is not an estimate of decay. The validation figure is the maximum of a ranking over more
than a thousand backtests, so it carries the selection; the holdout figure is one measurement
over a much shorter window, so it carries that window's sampling error. Both push the pair
apart on their own, before any real change in the strategy's edge.
[`20_strategy_analysis`](20_strategy_analysis.ipynb) is where they are given intervals and a
paired comparison.

<!-- ML4T_ZH_BEGIN c0010-p000 -->
## 4. 最終結果

下方兩個數字是同一策略在兩個互不重疊期間的量測，兩者差距並非衰減程度的估計。驗證期數字是超過一千次回測排名中的最大值，因此含有選擇效果；保留測試期數字則來自短得多的期間中的單次量測，因此含有該期間的抽樣誤差。即使策略優勢沒有真的改變，兩者也各自會拉大數字間的距離。[`20_strategy_analysis`](20_strategy_analysis.ipynb) 會提供區間估計及配對比較。
<!-- ML4T_ZH_END c0010-p000 -->


In [7]:
metrics = result.metrics
# The selected configuration's own registered Sharpe, not the resolver's. `resolve_solvent_carrier`
# reports the common-support figure, which re-ranks candidates on the timestamps every one of them
# covers; that is the right number for choosing between candidates and the wrong one to set beside a
# holdout measured over its own full window. Both are printed, so neither has to be inferred from
# the other.
with sqlite3.connect(str(CASE_DIR / "run_log" / "registry.db")) as conn:
    carrier_sharpe, carrier_periods, carrier_trades = conn.execute(
        "SELECT sharpe, n_periods, num_trades FROM backtest_metrics WHERE backtest_hash = ?",
        (carrier["val_backtest_hash"],),
    ).fetchone()

# `n_periods` is on the DAILY grid, not the decision grid. The backtester aggregates to daily
# returns before it computes anything, and `evaluation.periods_per_year` (252) annualizes that
# grid - which is why a six-month intraday holdout reports a few hundred periods rather than
# tens of thousands. Reading it as decision slots is the error that made the committed
# NASDAQ-100 benchmark understate itself roughly fivefold before #362 regenerated it.
print(f"Validation Sharpe over {int(carrier_periods):,} sessions: {carrier_sharpe:.3f}")
print(f"  the same run re-ranked on common support: {carrier['val_sharpe']:.3f}")
print(
    f"Holdout Sharpe over {int(metrics['n_periods']):,} sessions:    "
    f"{metrics.get('sharpe', float('nan')):.3f}"
)
print(
    f"Holdout: CAGR {metrics.get('cagr', float('nan')):.1%}, "
    f"max drawdown {metrics.get('max_drawdown', float('nan')):.2%}, "
    f"win rate {metrics.get('win_rate', float('nan')):.0%}"
)
# This case study runs the bar-by-bar engine - the configuration it overlays declares a trailing
# stop, which a vectorized weight-times-return path cannot express - so trade counts are recorded
# and can be compared. A holdout that rebalanced far less than the validation run at the same
# cadence would say the basket stopped changing, which is a different thing from a lower Sharpe.
print(
    f"Trades: {int(metrics.get('num_trades', 0)):,} on the holdout, "
    f"{int(carrier_trades):,} on validation"
)

Validation Sharpe over 253 sessions: 2.416
  the same run re-ranked on common support: 2.416
Holdout Sharpe over 128 sessions:    1.320
Holdout: CAGR 22.9%, max drawdown -10.94%, win rate 52%
Trades: 420 on the holdout, 900 on validation


## What this notebook establishes, and what it does not

It establishes a return series for the selected configuration over a period no choice in this
case study was made on. That is the only thing a holdout can give, and it is worth less than
it looks: the window is short beside the validation span it is being compared with, which is
too few observations to separate a strategy that decayed from one that had an ordinary year.

It does not establish that this configuration was the right one to carry here. The selection
that brought it was made on validation, over a pool large enough that its maximum is
optimistic by construction, and this notebook inherits that pool without correcting for it.
The deflation is [`20_strategy_analysis`](20_strategy_analysis.ipynb)'s.

Re-running this notebook against the same configuration is free and idempotent - the backtest hash
is unchanged and the registered run is served back. A different configuration is refused, for the
reason 18 gives.

**Next:** [`20_strategy_analysis`](20_strategy_analysis.ipynb).

<!-- ML4T_ZH_BEGIN c0012-p000 -->
## 本 Notebook 確立了什麼，又未確立什麼

它確立選定設定在本案例未曾用來作選擇的期間內，產生了一條報酬序列。這是保留測試期唯一能提供的資訊，其證據力比表面上看來弱：該期間遠短於用來比較的驗證期，觀測值不足以區分策略優勢衰減與一般年度波動。

它並未證明此設定是本案例應採用的正確選擇。帶來該設定的選擇發生在驗證期，候選池大到其最大值按設計就偏樂觀；本 Notebook 繼承該候選池，沒有修正此偏差。去膨脹處理由 [`20_strategy_analysis`](20_strategy_analysis.ipynb) 執行。

以相同設定重新執行本 Notebook 不增加成本，且具有冪等性：回測雜湊值不變，系統會回傳已登錄的執行結果。若設定不同，則會依第 18 份 Notebook 說明的理由拒絕。

**下一步：**[`20_strategy_analysis`](20_strategy_analysis.ipynb)。
<!-- ML4T_ZH_END c0012-p000 -->
